In [ ]:
import cv2
import time
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path

# Load your video file (replace with your video path)
VIDEO_PATH = "your_video.mp4"

def detect_objects_with_model(model_name, video_path):
    """
    Detect objects in video using specified YOLO model
    Returns: detection time, fps, and results
    """
    # Load model
    model = YOLO(model_name)
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    
    frame_count = 0
    total_time = 0
    results_list = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Perform detection
        start_time = time.time()
        results = model(frame, verbose=False)
        end_time = time.time()
        
        total_time += (end_time - start_time)
        frame_count += 1
        results_list.append(results[0])
        
        # Process only first 100 frames for comparison
        if frame_count >= 100:
            break
    
    cap.release()
    
    avg_time = total_time / frame_count
    fps = 1 / avg_time
    
    return {
        'model': model_name,
        'avg_detection_time': avg_time,
        'fps': fps,
        'frame_count': frame_count,
        'results': results_list
    }

# Test different YOLO models
models_to_test = ['yolov8n.pt', 'yolov11n.pt', 'yolo11n.pt']  # Use appropriate model names

comparison_results = []

for model_name in models_to_test:
    print(f"\nTesting {model_name}...")
    try:
        result = detect_objects_with_model(model_name, VIDEO_PATH)
        comparison_results.append(result)
        print(f"Model: {model_name}")
        print(f"Average Detection Time: {result['avg_detection_time']:.4f}s")
        print(f"FPS: {result['fps']:.2f}")
    except Exception as e:
        print(f"Error with {model_name}: {e}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = [r['model'] for r in comparison_results]
times = [r['avg_detection_time'] for r in comparison_results]
fps_values = [r['fps'] for r in comparison_results]

axes[0].bar(models, times, color=['blue', 'green', 'red'])
axes[0].set_ylabel('Detection Time (seconds)')
axes[0].set_title('Average Detection Time Comparison')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(models, fps_values, color=['blue', 'green', 'red'])
axes[1].set_ylabel('FPS')
axes[1].set_title('Frames Per Second Comparison')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('yolo_comparison.png')
plt.show()